# Korean PII STT Two-Stage Beep Pipeline v2

Colab T4 테스트용 노트북입니다.

반영 사항:
- `전라남도 목포시 통일대로 140` 같은 주소 패턴 강화
- `010-2345-2222` 중 `010`만 삐 처리되는 문제 수정
- 전화번호는 숫자 전체 기준으로 word timestamp 매칭
- 최종 삐 처리는 이름/주소/전화번호의 정밀 매칭 구간에만 적용

In [ ]:
# 1. 설치
!pip install -q faster-whisper transformers accelerate pydub
!apt-get install -y ffmpeg > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.3 MB/s eta 0:00:00


In [ ]:
# 2. import
import os
import re
import json
import torch
import subprocess

from difflib import SequenceMatcher
from google.colab import files
from faster_whisper import WhisperModel
from transformers import pipeline
from pydub import AudioSegment
from pydub.generators import Sine

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# 3. 영상 업로드
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("업로드된 영상이 없습니다.")
VIDEO_PATH = list(uploaded.keys())[0]
print(f"[UPLOAD] VIDEO_PATH = {VIDEO_PATH}")

Saving 개인정보4.mp4 to 개인정보4.mp4
[UPLOAD] VIDEO_PATH = 개인정보4.mp4


In [ ]:
# 4. 디바이스 설정
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
print("[DEVICE]", DEVICE)
print("[COMPUTE_TYPE]", COMPUTE_TYPE)

[DEVICE] cuda
[COMPUTE_TYPE] float16


In [ ]:
# 5. STT / NER 모델 로드
WHISPER_MODEL_SIZE = "large"  # 속도 우선이면 "small"
stt_model = WhisperModel(WHISPER_MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

NER_MODEL_NAME = "YakuzaNeko/kr-dlp-ner-roberta-large"
ner = pipeline(
    "token-classification",
    model=NER_MODEL_NAME,
    tokenizer=NER_MODEL_NAME,
    aggregation_strategy="simple",
    device=0 if DEVICE == "cuda" else -1
)
print("[MODEL] Loaded")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[MODEL] Loaded


In [ ]:
# 6. Regex 정의
PHONE_REGEX = re.compile(
    r"""
    (?:
        01[016789]
        \s*[-.]?\s*
        \d{3,4}
        \s*[-.]?\s*
        \d{4}
    )
    """,
    re.VERBOSE
)

ADDRESS_REGEX = re.compile(
    r"""
    (
        (?:
            서울특별시|서울시|부산광역시|부산시|대구광역시|대구시|인천광역시|인천시|
            광주광역시|광주시|대전광역시|대전시|울산광역시|울산시|세종특별자치시|세종시|
            경기도|강원특별자치도|강원도|충청북도|충북|충청남도|충남|전라북도|전북|
            전라남도|전남|경상북도|경북|경상남도|경남|제주특별자치도|제주도
        )
        \s*
        [가-힣]{1,20}(?:시|군|구)
        (?:\s*[가-힣]{1,20}(?:구|읍|면|동|리))?
        \s*
        [가-힣0-9]{1,30}(?:로|길|대로|번길)
        \s*
        \d{1,5}
        (?:-\d{1,5})?
    )
    """,
    re.VERBOSE
)

NAME_PATTERNS = [
    r"([가-힣]{2,4})\s*(?:입니다|이에요|예요)",
    r"([가-힣]{2,4})\s?(?:고객님|님|기사님|선생님)"
]

In [ ]:
# 7. 정규화 함수
def normalize_text(text: str) -> str:
    return re.sub(r"[\s\-\.\,\:\;]", "", text or "")

def only_digits(text: str) -> str:
    return re.sub(r"\D", "", text or "")

def clean_name_match(match):
    if match.groups():
        return match.group(1).strip()
    return match.group().strip()

In [ ]:
# 8. Regex 기반 개인정보 탐지
def detect_regex_entities(text: str):
    results = []
    for match in PHONE_REGEX.finditer(text):
        results.append({"type":"phone","text":match.group().strip(),"confidence":0.98,"source":"regex"})
    for match in ADDRESS_REGEX.finditer(text):
        results.append({"type":"address","text":match.group().strip(),"confidence":0.92,"source":"regex"})
    for pattern in NAME_PATTERNS:
        for match in re.finditer(pattern, text):
            name_text = clean_name_match(match)
            if len(name_text) >= 2:
                results.append({"type":"name","text":name_text,"confidence":0.82,"source":"regex"})
    return results

In [ ]:
# 9. NER 기반 개인정보 탐지
def map_ner_label(label: str):
    label = label.upper()
    if any(x in label for x in ["PER", "PERSON", "NAME"]):
        return "name"
    if any(x in label for x in ["LOC", "LOCATION", "ADDRESS", "ADDR"]):
        return "address"
    if any(x in label for x in ["PHONE", "TEL", "MOBILE"]):
        return "phone"
    return None

def detect_ner_entities(text: str):
    results = []
    try:
        ner_results = ner(text)
    except Exception as e:
        print("[NER ERROR]", e)
        return results
    for item in ner_results:
        entity_type = map_ner_label(item.get("entity_group", ""))
        if entity_type is None:
            continue
        word = item.get("word", "").strip()
        if not word:
            continue
        if entity_type == "phone" and len(only_digits(word)) < 9:
            continue
        results.append({"type":entity_type,"text":word,"confidence":float(item.get("score",0.0)),"source":"ner"})
    return results

In [ ]:
# 10. Word timestamp 정밀 매칭
FINAL_BEEP_TYPES = {"name", "phone", "address"}

def find_entity_time_by_words(entity_text, words, pad_sec=0.08):
    if not words:
        return None
    entity_norm = normalize_text(entity_text)
    if not entity_norm:
        return None
    max_window = min(12, len(words))
    best = None
    best_score = 0
    for size in range(1, max_window + 1):
        for i in range(0, len(words) - size + 1):
            chunk_words = words[i:i+size]
            chunk_text = "".join([w.word for w in chunk_words])
            chunk_norm = normalize_text(chunk_text)
            if not chunk_norm:
                continue
            score = SequenceMatcher(None, entity_norm, chunk_norm).ratio()
            if entity_norm in chunk_norm or chunk_norm in entity_norm:
                score = max(score, 0.95)
            if score > best_score:
                best_score = score
                best = {
                    "start": max(0, float(chunk_words[0].start) - pad_sec),
                    "end": float(chunk_words[-1].end) + pad_sec,
                    "matched_words": chunk_text,
                    "match_score": round(score, 4)
                }
    if best and best_score >= 0.72:
        return best
    return None

def find_phone_time_by_words(phone_text, words, pad_sec=0.08):
    if not words:
        return None
    target_digits = only_digits(phone_text)
    if not target_digits or len(target_digits) < 9:
        return None
    max_window = min(16, len(words))
    best = None
    best_score = 0
    for size in range(1, max_window + 1):
        for i in range(0, len(words) - size + 1):
            chunk_words = words[i:i+size]
            chunk_text = "".join([w.word for w in chunk_words])
            chunk_digits = only_digits(chunk_text)
            if not chunk_digits:
                continue
            if target_digits == chunk_digits:
                return {
                    "start": max(0, float(chunk_words[0].start) - pad_sec),
                    "end": float(chunk_words[-1].end) + pad_sec,
                    "matched_words": chunk_text,
                    "match_score": 1.0
                }
            score = SequenceMatcher(None, target_digits, chunk_digits).ratio()
            if target_digits in chunk_digits or chunk_digits in target_digits:
                score = max(score, min(len(target_digits), len(chunk_digits)) / len(target_digits))
            if score > best_score:
                best_score = score
                best = {
                    "start": max(0, float(chunk_words[0].start) - pad_sec),
                    "end": float(chunk_words[-1].end) + pad_sec,
                    "matched_words": chunk_text,
                    "match_score": round(score, 4)
                }
    if best and best_score >= 0.85:
        return best
    return None

In [ ]:
# 11. STT 실행
segments_iter, info = stt_model.transcribe(
    VIDEO_PATH,
    language="ko",
    word_timestamps=True,
    vad_filter=True
)
segments = list(segments_iter)
print("[STT DONE]")
print("language:", info.language)
print("duration:", round(info.duration, 2), "sec")
print("segment count:", len(segments))
for idx, seg in enumerate(segments[:10]):
    print(f"[{idx}] {seg.start:.2f} ~ {seg.end:.2f} | {seg.text}")

[STT DONE]
language: ko
duration: 29.56 sec
segment count: 4
[0] 0.69 ~ 8.45 |  유재석입니다. 최근 생성형 AI 기술이 급격히 발달하면서 기업들의 업무 자동화 도입이 과소화되고 있습니다.
[1] 9.13 ~ 12.61 |  전라남도 목포시 통일대로 140에 삽니다.
[2] 13.35 ~ 21.75 |  보고서 작성, 데이터 분석 등 단순 반복 업무의 효율성이 크게 향상된 반면 010-2345-2222입니다.
[3] 22.47 ~ 28.57 |  일자리 감수에 대한 직장인들의 고용 불안감과 보안 유출 우려도 함께 커지고 있습니다.


In [ ]:
# 12. 1차 후보 탐지 + 2차 정밀 위치 매칭
final_pii_items = []
for segment in segments:
    text = segment.text.strip()
    words = getattr(segment, "words", None)
    regex_entities = detect_regex_entities(text)
    ner_entities = detect_ner_entities(text)
    raw_entities = regex_entities + ner_entities
    final_entities = []
    for ent in raw_entities:
        ent_type = ent.get("type")
        ent_text = ent.get("text", "").strip()
        if ent_type not in FINAL_BEEP_TYPES or not ent_text:
            continue
        if ent_type == "phone":
            time_match = find_phone_time_by_words(ent_text, words)
        else:
            time_match = find_entity_time_by_words(ent_text, words)
        if time_match is None:
            continue
        final_entities.append({
            "type": ent_type,
            "text": ent_text,
            "start_time": round(time_match["start"], 2),
            "end_time": round(time_match["end"], 2),
            "confidence": round(float(ent.get("confidence", 0.0)), 4),
            "source": ent.get("source", "unknown"),
            "matched_words": time_match["matched_words"],
            "match_score": time_match["match_score"]
        })
    if final_entities:
        final_pii_items.append({
            "segment_start": round(segment.start, 2),
            "segment_end": round(segment.end, 2),
            "segment_text": text,
            "raw_entities": raw_entities,
            "entities": final_entities
        })
print("[DETECTION DONE]")
print("segments with pii:", len(final_pii_items))
print("total precise entities:", sum(len(x["entities"]) for x in final_pii_items))

[DETECTION DONE]
segments with pii: 3
total precise entities: 6


In [ ]:
# 13. 탐지 결과 확인
print("=" * 80)
print("[PII DETECTION RESULT]")
print("=" * 80)
for item in final_pii_items:
    print()
    print(f"[SEGMENT] {item['segment_start']} ~ {item['segment_end']}")
    print("TEXT:", item["segment_text"])
    print("RAW:")
    for raw in item["raw_entities"]:
        print("  -", raw)
    print("PRECISE:")
    for ent in item["entities"]:
        print(f"  -> {ent['type']} | {ent['text']} | {ent['start_time']} ~ {ent['end_time']} | matched={ent['matched_words']} | score={ent['match_score']} | source={ent['source']}")

[PII DETECTION RESULT]

[SEGMENT] 0.69 ~ 8.45
TEXT: 유재석입니다. 최근 생성형 AI 기술이 급격히 발달하면서 기업들의 업무 자동화 도입이 과소화되고 있습니다.
RAW:
  - {'type': 'name', 'text': '유재석', 'confidence': 0.82, 'source': 'regex'}
  - {'type': 'name', 'text': '유재석', 'confidence': 0.9962635636329651, 'source': 'ner'}
PRECISE:
  -> name | 유재석 | 0.61 ~ 1.71 | matched= 유재석입니다. | score=0.95 | source=regex
  -> name | 유재석 | 0.61 ~ 1.71 | matched= 유재석입니다. | score=0.95 | source=ner

[SEGMENT] 9.13 ~ 12.61
TEXT: 전라남도 목포시 통일대로 140에 삽니다.
RAW:
  - {'type': 'address', 'text': '전라남도 목포시 통일대로 140', 'confidence': 0.92, 'source': 'regex'}
  - {'type': 'address', 'text': '전라남도 목포시 통일대로 140', 'confidence': 0.9997316598892212, 'source': 'ner'}
PRECISE:
  -> address | 전라남도 목포시 통일대로 140 | 9.05 ~ 12.31 | matched= 전라남도 목포시 통일대로 140에 | score=0.9655 | source=regex
  -> address | 전라남도 목포시 통일대로 140 | 9.05 ~ 12.31 | matched= 전라남도 목포시 통일대로 140에 | score=0.9655 | source=ner

[SEGMENT] 13.35 ~ 21.75
TEXT: 보고서 작성, 데이터 분석 등 단순 반복 업무의 효율성이 크게 향상된 반면 010-2345-

In [ ]:
# 14. JSON 저장
JSON_OUTPUT = "/content/pii_precise_result.json"
with open(JSON_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(final_pii_items, f, ensure_ascii=False, indent=2)
print("[JSON SAVED]", JSON_OUTPUT)

[JSON SAVED] /content/pii_precise_result.json


In [ ]:
# 15. 원본 영상에서 오디오 추출
AUDIO_PATH = "/content/original_audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", VIDEO_PATH,
    "-vn", "-acodec", "pcm_s16le", "-ar", "44100", "-ac", "2", AUDIO_PATH
], check=True)
print("[AUDIO EXTRACTED]", AUDIO_PATH)

[AUDIO EXTRACTED] /content/original_audio.wav


In [ ]:
# 16. 정확한 개인정보 구간만 삐- 처리
audio = AudioSegment.from_wav(AUDIO_PATH)
BEEP_FREQ = 1000
BEEP_GAIN_DB = -8
beep_ranges = []
for item in final_pii_items:
    for ent in item["entities"]:
        start_ms = int(ent["start_time"] * 1000)
        end_ms = int(ent["end_time"] * 1000)
        duration_ms = max(0, end_ms - start_ms)
        if duration_ms <= 0:
            continue
        beep_ranges.append({
            "type": ent["type"],
            "text": ent["text"],
            "start_time": ent["start_time"],
            "end_time": ent["end_time"],
            "duration_ms": duration_ms
        })
        beep = Sine(BEEP_FREQ).to_audio_segment(duration=duration_ms).apply_gain(BEEP_GAIN_DB)
        beep = beep.set_frame_rate(audio.frame_rate)
        beep = beep.set_channels(audio.channels)
        beep = beep.set_sample_width(audio.sample_width)
        audio = audio[:start_ms] + beep + audio[end_ms:]
BEEP_AUDIO_PATH = "/content/beeped_audio.wav"
audio.export(BEEP_AUDIO_PATH, format="wav")
print("[BEEP DONE]")
print("beep count:", len(beep_ranges))
for r in beep_ranges:
    print(r)

[BEEP DONE]
beep count: 6
{'type': 'name', 'text': '유재석', 'start_time': 0.61, 'end_time': 1.71, 'duration_ms': 1100}
{'type': 'name', 'text': '유재석', 'start_time': 0.61, 'end_time': 1.71, 'duration_ms': 1100}
{'type': 'address', 'text': '전라남도 목포시 통일대로 140', 'start_time': 9.05, 'end_time': 12.31, 'duration_ms': 3260}
{'type': 'address', 'text': '전라남도 목포시 통일대로 140', 'start_time': 9.05, 'end_time': 12.31, 'duration_ms': 3260}
{'type': 'phone', 'text': '010-2345-2222', 'start_time': 18.87, 'end_time': 21.83, 'duration_ms': 2960}
{'type': 'phone', 'text': '010 - 2345 - 2222', 'start_time': 18.87, 'end_time': 21.83, 'duration_ms': 2960}


In [ ]:
# 17. 원본 영상 + 삐 처리 오디오 합성
OUTPUT_VIDEO = "/content/final_beeped_video.mp4"
subprocess.run([
    "ffmpeg", "-y", "-i", VIDEO_PATH, "-i", BEEP_AUDIO_PATH,
    "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-shortest", OUTPUT_VIDEO
], check=True)
print("[FINAL OUTPUT]")
print("VIDEO:", OUTPUT_VIDEO)
print("JSON :", JSON_OUTPUT)

[FINAL OUTPUT]
VIDEO: /content/final_beeped_video.mp4
JSON : /content/pii_precise_result.json


In [ ]:
# 18. 검증 리포트 저장
VERIFY_REPORT = {
    "video_path": VIDEO_PATH,
    "stt_segment_count": len(segments),
    "pii_segment_count": len(final_pii_items),
    "precise_entity_count": sum(len(x["entities"]) for x in final_pii_items),
    "beep_count": len(beep_ranges),
    "beep_ranges": beep_ranges,
    "output_video": OUTPUT_VIDEO,
    "json_output": JSON_OUTPUT
}
VERIFY_OUTPUT = "/content/verify_report.json"
with open(VERIFY_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(VERIFY_REPORT, f, ensure_ascii=False, indent=2)
print("[VERIFY SAVED]", VERIFY_OUTPUT)
print(json.dumps(VERIFY_REPORT, ensure_ascii=False, indent=2))

In [ ]:
# 19. 결과 다운로드
files.download(OUTPUT_VIDEO)
files.download(JSON_OUTPUT)
files.download(VERIFY_OUTPUT)